In [ ]:
!pip install "flwr[simulation]"

In [ ]:
from transformers import pipeline

# Load pre-trained NER pipeline
ner_pipeline = pipeline("ner", grouped_entities=True)

In [ ]:
# federated_lora_dual_unlearning_mia_adapters_only.py
# Flower-integrated federated LoRA with:
# - Heavy sensitive injection (emails + canaries)
# - Membership Inference Attack (MIA) before/after/unlearning
# - Dual-adapter unlearning on sensitive data
# - Adapter-only evaluation (norms, cosine sim, top-k drifts)
# Usage: python federated_lora_dual_unlearning_mia_adapters_only.py

import os
import re
import random
import json
import torch
import flwr as fl
import numpy as np
import pandas as pd
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model
from typing import List, Dict, Tuple

# -----------------------------
#  Repro & device
# -----------------------------
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using device: {device}")

# -----------------------------
# Paths
# -----------------------------
os.makedirs("client_data", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("final_global_model_2", exist_ok=True)

# -----------------------------
# Public dataset (Wikitext-2)
# -----------------------------
print("Loading public dataset (wikitext-2)...")
try:
    public_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
    public_texts = [t for t in public_dataset["train"]["text"] if isinstance(t, str) and t.strip()]
    print(f"Loaded {len(public_texts)} public texts.")
except Exception as e:
    print("Failed to load wikitext dataset (offline?). Using small fallback list.", e)
    public_texts = ["This is a fallback public sentence."] * 200

# -----------------------------
# Canary & email utilities
# -----------------------------
CANARY_TAG = "CANARY_SECRET"
email_regex = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")
canary_regex = re.compile(rf"{CANARY_TAG}_\d{{3}}_X[0-9A-Z]+")

def make_canaries(n_canaries=5000, token_len=8):
    canaries = []
    for i in range(n_canaries):
        canary = f"{CANARY_TAG}_{i:03d}_X" + "".join(random.choices("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789", k=token_len))
        # Add natural-looking context for stronger memorization cues
        canaries.append(f"User token: {canary}. Please remember this identifier.")
    return canaries

def make_synthetic_emails(n_emails=200):
    domains = ["example.com", "mail.test", "demo.org", "sample.net", "private.co"]
    names = ["alice", "bob", "carol", "dave", "eve", "mallory", "trent", "peggy", "victor", "walter"]
    emails = []
    for _ in range(n_emails):
        name = random.choice(names) + str(random.randint(100, 999))
        dom = random.choice(domains)
        e = f"{name}@{dom}"
        emails.append(f"Contact: {e}. This is sensitive; do not expose.")
    return emails
# Entities we consider sensitive
def detect_sensitive(text):
    if not isinstance(text, str) or not text.strip():
        return []

    found = set()

    # --------------------
    # Regex (high confidence)
    # --------------------
    emails = re.findall(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', text)
    phones = re.findall(r'\b(?:\+?\d{1,3})?[\s-]?(?:\d{2,3}[\s-]?){2,4}\d{2,4}\b', text)

    for e in emails + phones:
        found.add(e)

    # --------------------
    # NER (semantic, filtered)
    # --------------------
    entities = ner_pipeline(text)
    for e in entities:
        ent_type = e["entity_group"]
        word = e["word"].replace("##", "").strip()

        # Only strong identifiers
        if ent_type == "PER" and len(word) > 3:
            found.add(word)

    return list(found)

def redact_sensitive(text):
    sensitive_entities = detect_sensitive(text)
    redacted_text = text
    for entity in sensitive_entities:
        entity_pattern = re.escape(entity)
        redacted_text = re.sub(entity_pattern, "[REDACTED]", redacted_text, flags=re.IGNORECASE)
    return redacted_text

def redact_dataset(texts):
    return [redact_sensitive(t) for t in texts]

# -----------------------------
#  Tokenizer & LoRA model factory (separate adapters)
# -----------------------------
model_name = "EleutherAI/gpt-neo-125M"
print(f"Loading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def create_lora_model():
    base = AutoModelForCausalLM.from_pretrained(model_name, low_cpu_mem_usage=True)
    base.config.pad_token_id = tokenizer.eos_token_id

    # Freeze base parameters
    for _, p in base.named_parameters():
        p.requires_grad = False

    # LoRA adapter configs
    lora_config_forget = LoraConfig(
        r=16, lora_alpha=64, target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    lora_config_retain = LoraConfig(
        r=16, lora_alpha=64, target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )

    # Create peft model (initial adapter)
    model = get_peft_model(base, lora_config_forget)

    # Register adapters robustly
    def try_add_adapter(m, name, cfg):
        try:
            m.add_adapter(name, cfg)
            return True
        except Exception:
            pass
        try:
            m.add_adapter(name, cfg)
            return True
        except Exception:
            pass
        try:
            if hasattr(m, "peft_config") and isinstance(m.peft_config, dict):
                if name not in m.peft_config:
                    m.peft_config[name] = cfg
                    return True
        except Exception:
            pass
        return False

    _ = try_add_adapter(model, "forget", lora_config_forget)
    _ = try_add_adapter(model, "retain", lora_config_retain)

    # Set active adapter if possible
    try:
        model.set_adapter("forget")
    except Exception:
        try:
            model.active_adapter = "forget"
        except Exception:
            pass

    named_keys = list(model.state_dict().keys())
    forget_keys = [k for k in named_keys if "forget" in k] or [k for k in named_keys if "lora" in k]
    retain_keys = [k for k in named_keys if "retain" in k]

    for _, p in model.named_parameters():
        p.requires_grad = False

    model.to(device)
    model._adapter_param_keys = {"forget": set(forget_keys), "retain": set(retain_keys)}
    print(f"create_lora_model: found {len(forget_keys)} forget keys, {len(retain_keys)} retain keys")
    return model

def get_adapter_param_names(model, adapter_name):
    if hasattr(model, "_adapter_param_keys") and model._adapter_param_keys.get(adapter_name):
        return list(model._adapter_param_keys[adapter_name])
    # Fallback
    all_names = [n for n, _ in model.named_parameters()]
    keys = [n for n in all_names if adapter_name in n]
    if keys:
        return keys
    return [n for n in all_names if "lora" in n]

def tokenize_function(texts):
    if isinstance(texts, str):
        texts = [texts]
    return tokenizer(texts, padding=True, truncation=True, max_length=64)

# -----------------------------
# Build client splits: public-only and sensitive-injected
# -----------------------------
num_clients = 3
random.shuffle(public_texts)
public_size_target = 1200 if len(public_texts) >= 1200 else len(public_texts)
public_texts_sub = public_texts[:public_size_target]

# Public-only splits
client_splits_public = [[] for _ in range(num_clients)]
shard = max(1, len(public_texts_sub) // num_clients)
for i in range(num_clients):
    start = i * shard
    end = start + shard
    client_splits_public[i].extend(public_texts_sub[start:end])

# Heavy injection: synthetic emails + canaries everywhere
synthetic_emails = make_synthetic_emails(n_emails=300)
canaries = make_canaries(n_canaries=80, token_len=10)
sensitive_payload = synthetic_emails + canaries

# Sensitive-injected splits: start from public-only, then add sensitive
client_splits_sensitive = [list(s) for s in client_splits_public]
for payload in sensitive_payload:
    # Inject into every client with high probability to simulate "everywhere"
    for cid in range(num_clients):
        if random.random() < 0.9:
            client_splits_sensitive[cid].append(payload)

# Balance lengths per setting
def balance_splits(splits):
    min_len = min(len(s) for s in splits) if splits else 0
    return [s[:min_len] for s in splits]

client_splits_public = balance_splits(client_splits_public)
client_splits_sensitive = balance_splits(client_splits_sensitive)

print(f"Public-only client lengths: {[len(s) for s in client_splits_public]}")
print(f"Sensitive-injected client lengths: {[len(s) for s in client_splits_sensitive]}")

# Save splits
for cid in range(num_clients):
    # Public-only split has no sensitive flagging
    client_texts_pub = client_splits_public[cid]
    sens_mask_pub = [len(detect_sensitive(t)) > 0 for t in client_texts_pub]
    forget_pub = [t for t, m in zip(client_texts_pub, sens_mask_pub) if m]  # likely empty
    retain_pub = [t for t, m in zip(client_texts_pub, sens_mask_pub) if not m]
    torch.save({"forget": forget_pub, "retain": retain_pub}, f"client_data/client{cid}_split_public.pt")

    # Sensitive-injected split
    client_texts_sens = client_splits_sensitive[cid]
    sens_mask_sens = [len(detect_sensitive(t)) > 0 for t in client_texts_sens]
    forget_sens = [t for t, m in zip(client_texts_sens, sens_mask_sens) if m]
    retain_sens = [t for t, m in zip(client_texts_sens, sens_mask_sens) if not m]
    torch.save({"forget": forget_sens, "retain": retain_sens}, f"client_data/client{cid}_split_sensitive.pt")

    print(f"Client {cid}: public_only -> forget={len(forget_pub)}, retain={len(retain_pub)} | sensitive_injected -> forget={len(forget_sens)}, retain={len(retain_sens)}")

# -----------------------------
# LoRA helpers
# -----------------------------
def save_lora_state(model, path):
    lora_state = {k: v.cpu() for k, v in model.state_dict().items() if "lora" in k or "adapter" in k}
    torch.save(lora_state, path)

def load_lora_state_into_model(model, path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    lora_state = torch.load(path, map_location="cpu")
    model_state = model.state_dict()
    for k, v in lora_state.items():
        if k in model_state:
            model_state[k] = v.to(device)
    model.load_state_dict(model_state, strict=False)

def get_sorted_lora_keys_from_model(model):
    return sorted([k for k in model.state_dict().keys() if "lora" in k or "adapter" in k])

def flatten_adapter_state(state_dict: Dict[str, torch.Tensor]) -> torch.Tensor:
    # Concatenate all adapter tensors into a single 1D vector
    vecs = []
    for k, v in sorted(state_dict.items()):
        if isinstance(v, torch.Tensor):
            vecs.append(v.float().view(-1))
    if not vecs:
        return torch.zeros(1)
    return torch.cat(vecs)

def adapter_state(model) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if "lora" in k or "adapter" in k}

def adapter_metrics(before: Dict[str, torch.Tensor], after: Dict[str, torch.Tensor], topk=10) -> Dict[str, float]:
    keys = sorted(set(before.keys()) | set(after.keys()))
    drifts = []
    for k in keys:
        b = before.get(k, torch.zeros_like(after.get(k, torch.zeros(1))))
        a = after.get(k, torch.zeros_like(b))
        drifts.append((k, torch.norm(a.float() - b.float()).item()))
    drifts.sort(key=lambda x: x[1], reverse=True)
    vec_b = flatten_adapter_state(before)
    vec_a = flatten_adapter_state(after)
    norm_b = torch.norm(vec_b).item()
    norm_a = torch.norm(vec_a).item()
    cos = (torch.dot(vec_b, vec_a) / (torch.norm(vec_b) * torch.norm(vec_a) + 1e-8)).item() if norm_b > 0 and norm_a > 0 else float("nan")
    print(f" Adapter-only metrics: ||before||={norm_b:.4f}, ||after||={norm_a:.4f}, cosine={cos:.4f}")
    print(" Top-k parameter drifts (adapter keys):")
    for k, d in drifts[:topk]:
        print(f"  {k}: Δnorm={d:.6f}")
    return {"norm_before": norm_b, "norm_after": norm_a, "cosine": cos}

# -----------------------------
# 4️⃣ LoRAClient with adapter-aware training + unlearning + MIA
# -----------------------------
class LoRAClient(fl.client.NumPyClient):
    def __init__(self, client_id, data):
        self.client_id = client_id
        self.data = data
        print(f"🚀 Client {client_id} initialized on {device} with {len(data)} samples")
        self.model = create_lora_model()
        self.lora_keys = get_sorted_lora_keys_from_model(self.model)
        self.device = device
        self.tokenizer = tokenizer

    def get_parameters(self, config):
        state = self.model.state_dict()
        arrays = [state[k].cpu().numpy() for k in self.lora_keys]
        return arrays

    def set_parameters(self, parameters):
        state = self.model.state_dict()
        if len(parameters) != len(self.lora_keys):
            raise ValueError(f"Expected {len(self.lora_keys)} params, got {len(parameters)}")
        for k, v in zip(self.lora_keys, parameters):
            state[k] = torch.tensor(v).to(device)
        self.model.load_state_dict(state, strict=False)

    # Adapter-aware local training
    def train_local(self, texts, epochs=1, lr=5e-5):
        if not texts:
            print(f"Client {self.client_id}: no local data to train.")
            return

        dataset = Dataset.from_dict({"text": texts})
        dataset = dataset.map(lambda x: tokenize_function(x["text"]), batched=True, remove_columns=["text"])
        dataset = dataset.train_test_split(test_size=0.1, seed=seed)
        data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

        args = TrainingArguments(
            output_dir=f"./results_client_{self.client_id}",
            per_device_train_batch_size=1,
            num_train_epochs=epochs,
            learning_rate=lr,
            logging_steps=50,
            save_total_limit=1,
            report_to="none",
            fp16=torch.cuda.is_available(),
        )

        for n, p in self.model.named_parameters():
            if "lora" in n or "adapter" in n:
                p.requires_grad = True
            else:
                p.requires_grad = False

        trainer = Trainer(
            model=self.model,
            args=args,
            train_dataset=dataset["train"],
            eval_dataset=dataset["test"],
            tokenizer=tokenizer,
            data_collator=data_collator,
        )

        trainer.train()
        save_path = f"checkpoints/client{self.client_id}_lora_last.pth"
        save_lora_state(self.model, save_path)
        print(f"Client {self.client_id} saved LoRA checkpoint -> {save_path}")

    # Lightweight single-sample loss (no Trainer) for MIA speed
    def loss_for_text(self, text: str) -> float:
        self.model.eval()
        with torch.no_grad():
            inputs = self.tokenizer([text], return_tensors="pt", padding=True, truncation=True, max_length=64).to(self.device)
            out = self.model(**inputs, labels=inputs["input_ids"])
            loss = float(out.loss.detach().cpu().item())
        return loss

    # Membership Inference Attack (loss thresholding)
    def run_mia(self, member_texts: List[str], nonmember_texts: List[str], threshold: float = None) -> Dict[str, float]:
        member_losses = [self.loss_for_text(t) for t in member_texts]
        nonmember_losses = [self.loss_for_text(t) for t in nonmember_texts]
        if threshold is None:
            threshold = (np.mean(member_losses) + np.mean(nonmember_losses)) / 2.0

        member_preds = [l < threshold for l in member_losses]
        nonmember_preds = [l < threshold for l in nonmember_losses]

        tp = sum(member_preds)
        fn = len(member_preds) - tp
        tn = sum(not p for p in nonmember_preds)
        fp = len(nonmember_preds) - tn
        acc = (tp + tn) / (len(member_preds) + len(nonmember_preds) + 1e-8)
        print(f" MIA: threshold={threshold:.4f}, acc={acc:.3f}, TP={tp}, FP={fp}, FN={fn}, TN={tn}")
        return {
            "threshold": threshold,
            "acc": acc,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            "member_mean_loss": float(np.mean(member_losses)) if member_losses else float("nan"),
            "nonmember_mean_loss": float(np.mean(nonmember_losses)) if nonmember_losses else float("nan")
        }

    # Dual-target unlearning (adapter-aware + optional DP)
    def unlearn_sensitive_tokens_dual(
        self,
        alpha=800.0,
        beta=1000.0,
        lr=5e-3,
        steps=50,
        batch_size=16,
        dp_clip=1000.0,
        dp_noise_multiplier=0.0
    ):
        split_path = f"client_data/client{self.client_id}_split_sensitive.pt"
        if not os.path.exists(split_path):
            print(f" Client {self.client_id} sensitive split not found.")
            return

        split_data = torch.load(split_path)
        forget_texts = split_data.get("forget", [])
        retain_texts = split_data.get("retain", [])
        retain_texts_redacted = redact_dataset(retain_texts) if retain_texts else []

        if not forget_texts:
            print(f" No forget data for client {self.client_id}. Skipping.")
            return

        forget_param_names = get_adapter_param_names(self.model, "forget")
        retain_param_names = get_adapter_param_names(self.model, "retain")
        all_param_names = set(forget_param_names) | set(retain_param_names)
        name_to_param = {n: p for n, p in self.model.named_parameters() if n in all_param_names}

        for p in name_to_param.values():
            p.requires_grad = True

        optimizer = torch.optim.AdamW([p for p in name_to_param.values()], lr=lr)
        self.model.train()

        for step in range(steps):
            optimizer.zero_grad()

            # Forget pass
            try:
                self.model.set_adapter("forget")
            except Exception:
                self.model.active_adapter = "forget"

            forget_batch = random.sample(forget_texts, min(batch_size, len(forget_texts)))
            forget_inputs = self.tokenizer(forget_batch, return_tensors="pt", padding=True, truncation=True, max_length=64).to(self.device)
            outputs_forget = self.model(**forget_inputs, labels=forget_inputs["input_ids"])
            loss_forget = outputs_forget.loss
            loss_forget.backward()
            forget_grads = {n: name_to_param[n].grad.detach().clone() for n in forget_param_names if name_to_param[n].grad is not None}
            optimizer.zero_grad()

            # Retain pass
            retain_grads = {}
            loss_retain = torch.tensor(0.0, device=self.device)
            if retain_texts_redacted:
                try:
                    self.model.set_adapter("retain")
                except Exception:
                    self.model.active_adapter = "retain"
                retain_batch = random.sample(retain_texts_redacted, min(batch_size, len(retain_texts_redacted)))
                retain_inputs = self.tokenizer(retain_batch, return_tensors="pt", padding=True, truncation=True, max_length=64).to(self.device)
                outputs_retain = self.model(**retain_inputs, labels=retain_inputs["input_ids"])
                loss_retain = outputs_retain.loss
                loss_retain.backward()
                retain_grads = {n: name_to_param[n].grad.detach().clone() for n in retain_param_names if name_to_param[n].grad is not None}
                optimizer.zero_grad()

            # Combine grads
            combined = {}
            for name in all_param_names:
                gf = forget_grads.get(name, torch.zeros_like(name_to_param[name]))
                gr = retain_grads.get(name, torch.zeros_like(name_to_param[name]))
                combined[name] = (-alpha) * gf + beta * gr

            # Clip + DP noise
            try:
                total_norm = torch.norm(torch.stack([g.norm(2) for g in combined.values()]))
            except Exception:
                total_norm = torch.tensor(0.0, device=self.device)
            clip_coef = float(dp_clip) / (float(total_norm) + 1e-6)
            for name, g in combined.items():
                p = name_to_param[name]
                g_clipped = g * clip_coef if clip_coef < 1.0 else g
                if dp_noise_multiplier > 0.0:
                    noise = torch.randn_like(g_clipped) * (dp_noise_multiplier * float(dp_clip))
                    g_clipped = g_clipped + noise
                p.grad = g_clipped

            if step % 5 == 0:
                print(f"Step {step:02d} | ForgetLoss={float(loss_forget):.4f} | RetainLoss={float(loss_retain):.4f} | TotalGradNorm={float(total_norm):.4f}")
            optimizer.step()

        # Save adapter after unlearning
        save_path = f"checkpoints/client{self.client_id}_lora_unlearn_last.pth"
        save_lora_state(self.model, save_path)
        print(f"Client {self.client_id} saved unlearned LoRA checkpoint -> {save_path}")

# -----------------------------
# Flower client factory
# -----------------------------
def client_fn(cid: str):
    cid_int = int(cid)
    # Default uses public-only for initial sim; we’ll manually run sensitive phase later
    return LoRAClient(cid_int, client_splits_public[cid_int])

# -----------------------------
# Federated simulation (public-only baseline)
# -----------------------------
client_resources = {"num_cpus": 1, "num_gpus": 0.1}
strategy = fl.server.strategy.FedAvg()

print("Starting Flower simulation (public-only baseline)...")
history = fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=num_clients,
    client_resources=client_resources,
    config=fl.server.ServerConfig(num_rounds=2),
    strategy=strategy,
)

# -----------------------------
# Baseline local training to generate initial LoRA checkpoints (public-only)
# -----------------------------
print("\n Baseline training (public-only) to generate LoRA checkpoints...")
for cid in range(num_clients):
    c = client_fn(str(cid))
    split = torch.load(f"client_data/client{cid}_split_public.pt")
    all_texts = split.get("forget", []) + split.get("retain", [])
    if all_texts:
        c.train_local(all_texts, epochs=1, lr=5e-5)
    else:
        print(f" Client {cid} has no data to train on.")

# -----------------------------
# 8️⃣ MIA baseline (should be near chance without sensitive data)
# -----------------------------
print("\n MIA baseline (public-only):")
for cid in range(num_clients):
    c = client_fn(str(cid))
    ckpt = f"checkpoints/client{cid}_lora_last.pth"
    if os.path.exists(ckpt):
        load_lora_state_into_model(c.model, ckpt)
    split_pub = torch.load(f"client_data/client{cid}_split_public.pt")
    # Members: take a subset of retain_pub (it was trained on)
    member_texts = split_pub.get("retain", [])[:50]
    # Non-members: held-out from public_texts not used (simple sample here)
    nonmember_texts = public_texts[public_size_target:public_size_target+50]
    mia_res = c.run_mia(member_texts, nonmember_texts)
    print(f"Client {cid} MIA baseline -> acc={mia_res['acc']:.3f}, member_loss={mia_res['member_mean_loss']:.3f}, nonmember_loss={mia_res['nonmember_mean_loss']:.3f}")

# -----------------------------
# 9️⃣ Sensitive phase: retrain with heavy injection to induce leakage
# -----------------------------
print("\n Sensitive phase training (heavy injection) to induce leakage...")
for cid in range(num_clients):
    c = client_fn(str(cid))
    split_sens = torch.load(f"client_data/client{cid}_split_sensitive.pt")
    all_texts_sens = split_sens.get("forget", []) + split_sens.get("retain", [])
    if all_texts_sens:
        c.train_local(all_texts_sens, epochs=1, lr=5e-5)
    else:
        print(f"⚠️ Client {cid} has no sensitive-injected data to train on.")

# -----------------------------
# 🔎 Capture adapter states after sensitive training (for adapter-only evaluation)
# -----------------------------
adapter_snapshots_sensitive = {}
for cid in range(num_clients):
    c = client_fn(str(cid))
    ckpt = f"checkpoints/client{cid}_lora_last.pth"
    if os.path.exists(ckpt):
        load_lora_state_into_model(c.model, ckpt)
        adapter_snapshots_sensitive[cid] = adapter_state(c.model)

# -----------------------------
#  MIA after sensitive training (should succeed better)
# -----------------------------
print("\n MIA after sensitive training:")
for cid in range(num_clients):
    c = client_fn(str(cid))
    ckpt = f"checkpoints/client{cid}_lora_last.pth"
    if os.path.exists(ckpt):
        load_lora_state_into_model(c.model, ckpt)
    split_sens = torch.load(f"client_data/client{cid}_split_sensitive.pt")
    member_texts = split_sens.get("forget", [])[:100]  # sensitive members
    # Non-members: use public holdout not included in any split
    nonmember_texts = public_texts[public_size_target:public_size_target+100]
    mia_res = c.run_mia(member_texts, nonmember_texts)
    print(f"Client {cid} MIA post-leak -> acc={mia_res['acc']:.3f}, member_loss={mia_res['member_mean_loss']:.3f}, nonmember_loss={mia_res['nonmember_mean_loss']:.3f}")

# -----------------------------
# 10️⃣ Unlearning sensitive data (dual adapters)
# -----------------------------
print("\n Unlearning sensitive data (dual adapters)...")
for cid in range(num_clients):
    c = client_fn(str(cid))
    ckpt = f"checkpoints/client{cid}_lora_last.pth"
    if os.path.exists(ckpt):
        load_lora_state_into_model(c.model, ckpt)
    c.unlearn_sensitive_tokens_dual(
        alpha=800.0,
        beta=1000.0,
        lr=5e-3,
        steps=50,
        batch_size=16,
        dp_clip=1000.0,
        dp_noise_multiplier=0.0
    )

# -----------------------------
# 🔎 Capture adapter states after unlearning
# -----------------------------
adapter_snapshots_unlearned = {}
for cid in range(num_clients):
    c = client_fn(str(cid))
    ckpt_unlearn = f"checkpoints/client{cid}_lora_unlearn_last.pth"
    if os.path.exists(ckpt_unlearn):
        load_lora_state_into_model(c.model, ckpt_unlearn)
        adapter_snapshots_unlearned[cid] = adapter_state(c.model)

# -----------------------------
# 🔐 MIA after unlearning (should drop toward chance)
# -----------------------------
print("\n MIA after unlearning:")
for cid in range(num_clients):
    c = client_fn(str(cid))
    ckpt_unlearn = f"checkpoints/client{cid}_lora_unlearn_last.pth"
    if os.path.exists(ckpt_unlearn):
        load_lora_state_into_model(c.model, ckpt_unlearn)
    split_sens = torch.load(f"client_data/client{cid}_split_sensitive.pt")
    member_texts = split_sens.get("forget", [])[:100]  # previously sensitive members
    nonmember_texts = public_texts[public_size_target:public_size_target+100]
    mia_res = c.run_mia(member_texts, nonmember_texts)
    print(f"Client {cid} MIA post-unlearn -> acc={mia_res['acc']:.3f}, member_loss={mia_res['member_mean_loss']:.3f}, nonmember_loss={mia_res['nonmember_mean_loss']:.3f}")

# -----------------------------
#  Adapter-only evaluation: norms, cosine sim, top-k drifts
# -----------------------------
print("\n Adapter-only evaluation (drift from sensitive-trained -> unlearned):")
for cid in range(num_clients):
    sens = adapter_snapshots_sensitive.get(cid, {})
    unl = adapter_snapshots_unlearned.get(cid, {})
    if sens and unl:
        _ = adapter_metrics(sens, unl, topk=10)
    else:
        print(f"Client {cid}: missing adapter snapshots for evaluation.")

# -----------------------------
# 12️⃣ Save final global LoRA model (average client adapters)
# -----------------------------
ckpt_paths = [f"checkpoints/client{cid}_lora_unlearn_last.pth" for cid in range(num_clients)]
existing_ckpts = [p for p in ckpt_paths if os.path.exists(p)]
if not existing_ckpts:
    print(" No unlearned client LoRA checkpoints found to average. Skipping final averaging.")
else:
    lora_ckpts = [torch.load(p, map_location="cpu") for p in existing_ckpts]
    all_keys = sorted(set().union(*[set(x.keys()) for x in lora_ckpts]))
    avg_state = {}
    for k in all_keys:
        stacked = torch.stack([ckpt[k].float() for ckpt in lora_ckpts if k in ckpt], dim=0)
        avg_state[k] = stacked.mean(dim=0).to(device)

    final_global_model = create_lora_model()
    final_state = final_global_model.state_dict()
    for k, v in avg_state.items():
        if k in final_state:
            final_state[k] = v
    final_global_model.load_state_dict(final_state, strict=False)
    final_global_model.eval()
    print(" Final global LoRA model ready.")
    save_dir = "./final_global_model_2"
    final_global_model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f" Final global LoRA model and tokenizer saved to {save_dir}")

In [ ]:

# federated_lora_dual_unlearning_optionA_separate_adapters_fixed.py
# Corrected full script (Flower-integrated, separate forget/retain LoRA adapters,
# combined-grad unlearning with DP on forget, robust PEFT fallbacks).

import os
import re
import random
import math
import copy
import json
import torch
import flwr as fl
import numpy as np
import pandas as pd
from contextlib import nullcontext
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model

# -----------------------------
# Repro & device
# -----------------------------
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------------
# Paths
# -----------------------------
os.makedirs("client_data", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("final_global_model_2", exist_ok=True)

# -----------------------------
# (Optional) Load sensitive CSV (best-effort)
# -----------------------------
sensitive_csv_path = "/content/emaildata_100000_0.csv"
if os.path.exists(sensitive_csv_path):
    try:
        sensitive_df = pd.read_csv(sensitive_csv_path, on_bad_lines='skip', engine='python')
        sensitive_texts_full = sensitive_df.get("text", pd.Series()).dropna().astype(str).tolist()
        print(f"Loaded {len(sensitive_texts_full)} sensitive texts from CSV.")
    except Exception as e:
        print("Failed to load CSV:", e)
        sensitive_texts_full = []
else:
    sensitive_texts_full = []

# -----------------------------
# Public dataset (Wikitext-2)
# -----------------------------
print("Loading public dataset (wikitext-2)...")
try:
    public_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
    public_texts = [t for t in public_dataset["train"]["text"] if isinstance(t, str) and t.strip()]
    print(f"Loaded {len(public_texts)} public texts.")
except Exception as e:
    print("Failed to load wikitext dataset (offline?). Using small fallback list.", e)
    public_texts = ["This is a fallback public sentence."] * 200

# -----------------------------
# Canary utilities
# -----------------------------
CANARY_TAG = "CANARY_SECRET"

def make_canaries(n_canaries=20, token_len=6):
    canaries = []
    for i in range(n_canaries):
        canary = f" {CANARY_TAG}_{i:03d}_X" + "".join(random.choices("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789", k=token_len))
        canaries.append(canary)
    return canaries

def scatter_canaries_to_clients(client_splits, canaries, duplicate_prob=0.3):
    num_clients = len(client_splits)
    for c in canaries:
        cid = random.randrange(num_clients)
        client_splits[cid].append(c + " --context")
        for _ in range(num_clients - 1):
            if random.random() < duplicate_prob:
                cid2 = random.randrange(num_clients)
                client_splits[cid2].append(c + " duplicate context")
    return client_splits

# -----------------------------
# 2️⃣ Tokenizer & LoRA model factory (separate adapters)
# -----------------------------
model_name = "EleutherAI/gpt-neo-125M"
print(f"Loading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- Robust create_lora_model() replacement ---
from peft import PeftModel, PeftConfig

def _ensure_adapter_registered(model, name, config):
    """
    Ensure the adapter `name` is registered in the model's peft config.
    Tries add_adapter() in several ways; falls back to inserting into model.peft_config
    if the PEFT version exposes peft_config as a dict-like object.
    """
    # Try direct add_adapter API (common)
    try:
        model.add_adapter(name, config)
        return True
    except Exception:
        pass

    # Some PEFT versions expect positional signature model.add_adapter(name, config_obj)
    try:
        model.add_adapter(name, config)  # try again in case signature differs
        return True
    except Exception:
        pass

    # Fallback: try to inject into peft_config dict if available
    try:
        if hasattr(model, "peft_config"):
            pc = model.peft_config
            # if it's a dict-like mapping we can inject
            if isinstance(pc, dict):
                pc[name] = config
                model.peft_config = pc
                return True
            # if it's a PeftConfig-like object, attempt to read/modify its internal mapping if available
            # (best-effort; non-destructive)
            if isinstance(pc, PeftConfig):
                # Some versions use a dict field named `adapters` or similar — don't overwrite unknown internals
                # but try a safe route: set attribute if it exists
                try:
                    # create a shallow mapping if possible
                    if not hasattr(model, "_adapter_param_keys"):
                        # nothing to do here, but avoid error
                        pass
                    return False
                except Exception:
                    pass
    except Exception:
        pass

    return False


def create_lora_model():
    """
    Robust LoRA model factory. Uses module-level `device` and `tokenizer`.
    Ensures adapter registration for 'forget' and 'retain' and returns
    a model with model._adapter_param_keys = {"forget": set(...), "retain": set(...)}.
    """
    # Load base model
    base = AutoModelForCausalLM.from_pretrained(model_name, low_cpu_mem_usage=True)
    base.config.pad_token_id = tokenizer.eos_token_id

    # Freeze base parameters
    for n, p in base.named_parameters():
        p.requires_grad = False

    # LoRA adapter configs
    lora_config_forget = LoraConfig(
        r=16, lora_alpha=64, target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    lora_config_retain = LoraConfig(
        r=16, lora_alpha=64, target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )

    # Create peft model (initial adapter will be created by get_peft_model)
    model = get_peft_model(base, lora_config_forget)

    # Robustly ensure both named adapters exist.
    # Try model.add_adapter where available, else attempt lightweight peft_config injection.
    def try_add_adapter(m, name, cfg):
        try:
            m.add_adapter(name, cfg)
            return True
        except Exception:
            pass
        try:
            # Some PEFT versions accept a different signature; try again
            m.add_adapter(name, cfg)
            return True
        except Exception:
            pass
        # Fallback: if m.peft_config is a dict-like, inject keys (best-effort)
        try:
            if hasattr(m, "peft_config") and isinstance(m.peft_config, dict):
                if name not in m.peft_config:
                    m.peft_config[name] = cfg
                    return True
        except Exception:
            pass
        return False

    _ = try_add_adapter(model, "forget", lora_config_forget)
    _ = try_add_adapter(model, "retain", lora_config_retain)

    # Try to set the active adapter to 'forget' if available; otherwise leave as-is
    try:
        adapters = None
        if hasattr(model, "adapters"):
            adapters = getattr(model, "adapters")
        elif hasattr(model, "peft_config") and isinstance(model.peft_config, dict):
            adapters = list(model.peft_config.keys())
        elif hasattr(model, "list_adapters"):
            try:
                adapters = model.list_adapters()
            except Exception:
                adapters = None

        if adapters and "forget" in adapters:
            try:
                model.set_adapter("forget")
            except Exception:
                try:
                    model.active_adapter = "forget"
                except Exception:
                    pass
    except Exception:
        pass

    # Collect parameter names (state_dict keys)
    named_keys = list(model.state_dict().keys())
    forget_keys = [k for k in named_keys if "forget" in k]
    retain_keys = [k for k in named_keys if "retain" in k]

    # Fallbacks if explicit adapter-named keys not found
    if not forget_keys:
        forget_keys = [k for k in named_keys if "lora" in k]
        print("Warning: did not find explicit 'forget' keys; falling back to all lora keys for forget.")
    if not retain_keys:
        # if retain absent, leave retain_keys empty (we'll gracefully handle this)
        retain_keys = [k for k in named_keys if "retain" in k]

    # Ensure only LoRA/adapter params are trainable by default if desired.
    # Here we default to trainable=False for all, but leave LoRA params as requires_grad=False
    # so that caller controls which adapter is trainable per phase.
    for n, p in model.named_parameters():
        # By default freeze everything. User code will set requires_grad before training
        p.requires_grad = False

    model.to(device)

    # store adapter param name sets on the model for later lookups
    try:
        model._adapter_param_keys = {"forget": set(forget_keys), "retain": set(retain_keys)}
    except Exception:
        # Best-effort fallback if attribute cannot be set
        setattr(model, "_adapter_param_keys", {"forget": set(forget_keys), "retain": set(retain_keys)})

    # sanity prints (optional)
    print(f"create_lora_model: found {len(forget_keys)} forget keys, {len(retain_keys)} retain keys")
    return model

    # store adapter key lists on model for robust access later
    try:
        model._adapter_param_keys = {"forget": set(forget_keys), "retain": set(retain_keys)}
    except Exception:
        model._adapter_param_keys = {"forget": set(forget_keys), "retain": set(retain_keys)}
    return model

def get_adapter_param_names(model, adapter_name):
    """
    Return a list of parameter names that correspond to an adapter.
    This tries PEFT helper functions if available, else falls back to checking stored lists or parameter names.
    """
    # Try direct stored mapping
    if hasattr(model, "_adapter_param_keys") and model._adapter_param_keys.get(adapter_name):
        keys = list(model._adapter_param_keys[adapter_name])
        if keys:
            return keys

    # Try PEFT API: get_adapter_state_dict (some versions)
    try:
        if hasattr(model, "get_adapter_state_dict"):
            st = model.get_adapter_state_dict(adapter_name)
            if isinstance(st, dict) and st:
                return list(st.keys())
    except Exception:
        pass

    # Fallback: search names containing adapter_name (best-effort)
    all_names = [n for n, _ in model.named_parameters()]
    keys = [n for n in all_names if adapter_name in n]
    if keys:
        return keys

    # Fallback: give all lora keys
    keys = [n for n in all_names if "lora" in n]
    return keys

def tokenize_function(texts):
    # Use dynamic padding (padding=True) at batch-time rather than padding to max_length always
    if isinstance(texts, str):
        texts = [texts]
    return tokenizer(texts, padding=True, truncation=True, max_length=64)

# -----------------------------
# Simple detect_sensitive
# -----------------------------
email_regex = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")
canary_regex = re.compile(rf"{CANARY_TAG}_\d{{3}}_X[0-9A-Z]+")

# Entities we consider sensitive
def detect_sensitive(text):
    if not isinstance(text, str) or not text.strip():
        return []

    found = set()

    # --------------------
    # Regex (high confidence)
    # --------------------
    emails = re.findall(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', text)
    phones = re.findall(r'\b(?:\+?\d{1,3})?[\s-]?(?:\d{2,3}[\s-]?){2,4}\d{2,4}\b', text)

    for e in emails + phones:
        found.add(e)

    # --------------------
    # NER (semantic, filtered)
    # --------------------
    entities = ner_pipeline(text)
    for e in entities:
        ent_type = e["entity_group"]
        word = e["word"].replace("##", "").strip()

        # Only strong identifiers
        if ent_type == "PER" and len(word) > 3:
            found.add(word)

    return list(found)
def redact_sensitive(text):
    sensitive_entities = detect_sensitive(text)
    redacted_text = text
    for entity in sensitive_entities:
        entity_pattern = re.escape(entity)
        redacted_text = re.sub(entity_pattern, "[REDACTED]", redacted_text, flags=re.IGNORECASE)
    return redacted_text

def redact_dataset(texts):
    return [redact_sensitive(t) for t in texts]

# -----------------------------
# Prepare client splits
# -----------------------------
num_clients = 3
sensitive_size_target = min(1000, len(sensitive_texts_full)) if sensitive_texts_full else 0
public_size_target = sensitive_size_target if sensitive_size_target > 0 else 1000

random.shuffle(public_texts)
public_texts_sub = public_texts[:public_size_target]

client_splits = [[] for _ in range(num_clients)]
shard = max(1, len(public_texts_sub) // num_clients)
for i in range(num_clients):
    start = i * shard
    end = start + shard
    client_splits[i].extend(public_texts_sub[start:end])

# Inject sensitive texts (probabilistically per client)
for s in sensitive_texts_full[:sensitive_size_target]:
    for cid in range(num_clients):
        if random.random() < 0.6:
            client_splits[cid].append(s)

# Add canaries
num_canaries = 50
canaries = make_canaries(num_canaries, token_len=8)
client_splits = scatter_canaries_to_clients(client_splits, canaries, duplicate_prob=0.5)

# Balance lengths
min_len = min(len(s) for s in client_splits) if client_splits else 0
client_splits = [s[:min_len] for s in client_splits]

print(f"Client lengths: {[len(s) for s in client_splits]} | Canaries scattered: {len(canaries)}")

# -----------------------------
# Save per-client sensitive splits
# -----------------------------
for cid in range(num_clients):
    client_texts = client_splits[cid]
    sensitive_mask = [len(detect_sensitive(t)) > 0 for t in client_texts]
    forget_texts = [t for t, m in zip(client_texts, sensitive_mask) if m]
    retain_texts = [t for t, m in zip(client_texts, sensitive_mask) if not m]
    torch.save({"forget": forget_texts, "retain": retain_texts}, f"client_data/client{cid}_split.pt")
    print(f"Client {cid}: forget={len(forget_texts)}, retain={len(retain_texts)} saved")

# -----------------------------
# LoRA helpers
# -----------------------------
def save_lora_state(model, path):
    lora_state = {k: v.cpu() for k, v in model.state_dict().items() if "lora" in k or "adapter" in k}
    torch.save(lora_state, path)

def load_lora_state_into_model(model, path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    lora_state = torch.load(path, map_location="cpu")
    model_state = model.state_dict()
    for k, v in lora_state.items():
        if k in model_state:
            model_state[k] = v.to(device)
    model.load_state_dict(model_state, strict=False)

def get_sorted_lora_keys_from_model(model):
    return sorted([k for k in model.state_dict().keys() if "lora" in k or "adapter" in k])

# -----------------------------
# LoRAClient with adapter-aware training + separate unlearning
# -----------------------------
class LoRAClient(fl.client.NumPyClient):
    def __init__(self, client_id, data):
        self.client_id = client_id
        self.data = data
        print(f" Client {client_id} initialized on {device} with {len(data)} samples")
        self.model = create_lora_model()
        self.lora_keys = get_sorted_lora_keys_from_model(self.model)

        self.device = device
        self.tokenizer = tokenizer
    def get_parameters(self, config):
        state = self.model.state_dict()
        arrays = [state[k].cpu().numpy() for k in self.lora_keys]
        return arrays

    def set_parameters(self, parameters):
        state = self.model.state_dict()
        if len(parameters) != len(self.lora_keys):
            raise ValueError(f"Expected {len(self.lora_keys)} params, got {len(parameters)}")
        for k, v in zip(self.lora_keys, parameters):
            state[k] = torch.tensor(v).to(device)
        self.model.load_state_dict(state, strict=False)

    # REQUIRED by Flower: fit
    def fit(self, parameters, config):
        # Set incoming params
        self.set_parameters(parameters)

        # Local training: use a lightweight quick training call (1 epoch default)
        split = torch.load(f"client_data/client{self.client_id}_split.pt")
        all_texts = split.get("forget", []) + split.get("retain", [])
        if not all_texts:
            print(f"Client {self.client_id}: no data to fit. Returning current params.")
            return self.get_parameters({}), 0, {}

        epochs = int(config.get("local_epochs", 1)) if isinstance(config, dict) else 1
        self.train_local(all_texts, epochs=epochs, lr=5e-5)

        updated = self.get_parameters({})
        num_examples = len(all_texts)
        return updated, num_examples, {}

    # REQUIRED by Flower: evaluate
    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        split = torch.load(f"client_data/client{self.client_id}_split.pt")
        test_texts = split.get("retain", []) + split.get("forget", [])
        if not test_texts:
            print(f"Client {self.client_id}: no eval data.")
            return float("nan"), 0, {}
        loss, perp = self.evaluate_split(test_texts, label="server_eval")
        return float(loss), len(test_texts), {"perplexity": float(perp)}

    # -----------------
    # Adapter-aware local training
    # -----------------
    def train_local(self, texts, epochs=1, lr=5e-5):
        if not texts:
            print(f"Client {self.client_id}: no local data to train.")
            return

        dataset = Dataset.from_dict({"text": texts})
        dataset = dataset.map(lambda x: tokenize_function(x["text"]), batched=True, remove_columns=["text"])
        dataset = dataset.train_test_split(test_size=0.1, seed=seed)
        data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

        args = TrainingArguments(
            output_dir=f"./results_client_{self.client_id}",
            per_device_train_batch_size=1,
            num_train_epochs=epochs,
            learning_rate=lr,
            logging_steps=50,
            save_total_limit=1,
            report_to="none",
            fp16=torch.cuda.is_available(),
        )

        # Ensure LoRA params are trainable (both adapters)
        # We'll mark any parameter that contains 'lora' as trainable for this standard local training.
        for n, p in self.model.named_parameters():
            if "lora" in n or "adapter" in n:
                p.requires_grad = True
            else:
                p.requires_grad = False

        trainer = Trainer(
            model=self.model,
            args=args,
            train_dataset=dataset["train"],
            eval_dataset=dataset["test"],
            tokenizer=tokenizer,
            data_collator=data_collator,
        )

        trainer.train()
        save_path = f"checkpoints/client{self.client_id}_lora_last.pth"
        save_lora_state(self.model, save_path)
        print(f" Client {self.client_id} saved LoRA checkpoint -> {save_path}")

    # -----------------
    # Evaluate: returns NumPy-friendly loss/ppl
    # -----------------
    def evaluate_split(self, texts, label="eval"):
        loss, ppl = self.evaluate_texts_return(texts, label=label)
        return loss, ppl

    def evaluate_texts_return(self, texts, label="eval"):
        if not texts:
            print(f"Client {self.client_id}: no data for {label} evaluation.")
            return float("nan"), float("nan")
        dataset = Dataset.from_dict({"text": texts})
        dataset = dataset.map(lambda x: tokenize_function(x["text"]), batched=True, remove_columns=["text"])
        test_dataset = dataset.train_test_split(test_size=0.1, seed=seed)["test"]
        data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
        args = TrainingArguments(output_dir="./results_eval", per_device_eval_batch_size=2, report_to="none",
                                 fp16=torch.cuda.is_available())
        trainer = Trainer(model=self.model, args=args, eval_dataset=test_dataset,
                          tokenizer=tokenizer, data_collator=data_collator)
        eval_metrics = trainer.evaluate()
        loss = eval_metrics.get("eval_loss", float("nan"))
        try:
            perplexity = float(np.exp(loss))
        except Exception:
            perplexity = float("inf")
        print(f"📊 Client {self.client_id} - {label} | Loss: {loss:.4f}, Perplexity: {perplexity:.4f}")
        return float(loss), float(perplexity)

    # -----------------
    # Dual-target unlearning (adapter-aware + DP on forget)
    # -----------------
    def unlearn_sensitive_tokens_dual(
        self,
        alpha=800.0,     # very strong forget scaling
        beta=1000.0,         # weak retain guidance
        lr=5e-3,          # large step for visibility
        steps=10,         # reduced for quick debug
        batch_size=4,
        dp_clip=1000.0,
        dp_noise_multiplier=0.0
    ):
        # Load client split
        split_path = f"client_data/client{self.client_id}_split.pt"
        if not os.path.exists(split_path):
            print(f" Client {self.client_id} split not found.")
            return

        split_data = torch.load(split_path)
        forget_texts = split_data.get("forget", [])
        retain_texts = split_data.get("retain", [])
        retain_texts_redacted = redact_dataset(retain_texts) if retain_texts else []

        if not forget_texts:
            print(f" No forget data for client {self.client_id}. Skipping.")
            return

        # --- Get adapter params ---
        forget_param_names = get_adapter_param_names(self.model, "forget")
        retain_param_names = get_adapter_param_names(self.model, "retain")
        all_param_names = set(forget_param_names) | set(retain_param_names)
        name_to_param = {n: p for n, p in self.model.named_parameters() if n in all_param_names}

        if not forget_param_names:
            print(" Warning: forget_param_names is empty!")
        if not retain_param_names:
            print(" Warning: retain_param_names is empty!")

        # Ensure all are trainable
        for p in name_to_param.values():
            p.requires_grad = True

        optimizer = torch.optim.AdamW([p for p in name_to_param.values()], lr=lr)

        self.model.train()
        device = self.device
        tokenizer = self.tokenizer

        for step in range(steps):
            optimizer.zero_grad()

            # --- Forget pass ---
            try:
                self.model.set_adapter("forget")
            except Exception:
                self.model.active_adapter = "forget"

            forget_batch = random.sample(forget_texts, min(batch_size, len(forget_texts)))
            forget_inputs = tokenizer(forget_batch, return_tensors="pt", padding=True, truncation=True, max_length=64).to(device)
            outputs_forget = self.model(**forget_inputs, labels=forget_inputs["input_ids"])
            loss_forget = outputs_forget.loss
            loss_forget.backward()

            forget_grads = {n: name_to_param[n].grad.detach().clone() for n in forget_param_names if name_to_param[n].grad is not None}

            optimizer.zero_grad()

            # --- Retain pass ---
            retain_grads = {}
            if retain_texts_redacted:
                try:
                    self.model.set_adapter("retain")
                except Exception:
                    self.model.active_adapter = "retain"

                retain_batch = random.sample(retain_texts_redacted, min(batch_size, len(retain_texts_redacted)))
                retain_inputs = tokenizer(retain_batch, return_tensors="pt", padding=True, truncation=True, max_length=64).to(device)
                outputs_retain = self.model(**retain_inputs, labels=retain_inputs["input_ids"])
                loss_retain = outputs_retain.loss
                loss_retain.backward()
                retain_grads = {n: name_to_param[n].grad.detach().clone() for n in retain_param_names if name_to_param[n].grad is not None}
                optimizer.zero_grad()
            else:
                loss_retain = torch.tensor(0.0, device=device)

            # --- Combine grads with scaling ---
            combined = {}
            for name in all_param_names:
                gf = forget_grads.get(name, torch.zeros_like(name_to_param[name]))
                gr = retain_grads.get(name, torch.zeros_like(name_to_param[name]))
                combined[name] = (-alpha) * gf + beta * gr

            # --- Clip and add DP noise ---
            try:
                total_norm = torch.norm(torch.stack([g.norm(2) for g in combined.values()]))
            except Exception:
                total_norm = torch.tensor(0.0, device=device)

            clip_coef = float(dp_clip) / (float(total_norm) + 1e-6)
            for name, g in combined.items():
                p = name_to_param[name]
                g_clipped = g * clip_coef if clip_coef < 1.0 else g
                if dp_noise_multiplier > 0.0:
                    noise = torch.randn_like(g_clipped) * (dp_noise_multiplier * float(dp_clip))
                    g_clipped = g_clipped + noise
                p.grad = g_clipped

            # --- Debug: show norms before step ---
            print(f"Step {step:02d} | ForgetLoss={float(loss_forget):.4f} | RetainLoss={float(loss_retain):.4f} | TotalGradNorm={float(total_norm):.4f}")
            for n in list(forget_param_names)[:3]:  # print first 3 for brevity
                print(f"  {n}: grad norm={combined[n].norm():.6f}, param norm={name_to_param[n].data.norm():.6f}")

            # --- Step ---
            optimizer.step()

        # Save forget adapter for inspection
        save_path = f"checkpoints/client{self.client_id}_lora_unlearn_debug.pth"
        save_lora_state(self.model, save_path)
        print(f" Saved forget adapter -> {save_path}")



# -----------------------------
# Flower client factory
# -----------------------------
def client_fn(cid: str):
    cid_int = int(cid)
    return LoRAClient(cid_int, client_splits[cid_int])

# -----------------------------
#  Federated simulation
# -----------------------------
client_resources = {"num_cpus": 1, "num_gpus": 0.1}
strategy = fl.server.strategy.FedAvg()

print("Starting Flower simulation...")
history = fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=num_clients,
    client_resources=client_resources,
    config=fl.server.ServerConfig(num_rounds=2),
    strategy=strategy,
)

# -----------------------------
# Train clients locally first to generate checkpoints
# -----------------------------
print("\n Training clients locally to generate initial LoRA checkpoints...")
for cid in range(num_clients):
    c = client_fn(str(cid))
    split = torch.load(f"client_data/client{cid}_split.pt")
    all_texts = split.get("forget", []) + split.get("retain", [])
    if all_texts:
        c.train_local(all_texts, epochs=1, lr=5e-5)
    else:
        print(f"⚠️ Client {cid} has no data to train on.")

# -----------------------------
# Client-side unlearning & evaluation
# -----------------------------
for cid in range(num_clients):
    print("\n" + "="*60)
    print(f" Processing Client {cid}")
    c = client_fn(str(cid))
    ckpt = f"checkpoints/client{cid}_lora_last.pth"
    if os.path.exists(ckpt):
        try:
            load_lora_state_into_model(c.model, ckpt)
            print(f"Loaded checkpoint for client {cid}: {ckpt}")
        except Exception as e:
            print("Warning: failed to load client checkpoint:", e)
    else:
        print(f"No checkpoint found for client {cid} at {ckpt} (continuing with freshly initialized model)")

    split = torch.load(f"client_data/client{cid}_split.pt")
    forget_texts = split.get("forget", [])
    retain_texts = split.get("retain", [])

    # Evaluate BEFORE unlearning
    print(f"\n Client {cid} BEFORE unlearning evaluations:")
    forget_loss_before, forget_ppl_before = c.evaluate_texts_return(forget_texts, label="Forget_BEFORE")
    retain_loss_before, retain_ppl_before = c.evaluate_texts_return(retain_texts, label="Retain_BEFORE")
    combined_loss_before, combined_ppl_before = c.evaluate_texts_return(forget_texts + retain_texts, label="Combined_BEFORE")

    # Run unlearning
    print(f"\n Running unlearning for client {cid} ...")
    c.unlearn_sensitive_tokens_dual(
        alpha=800.0,          # very strong forget signal
        beta=1000,              # minimal retain guidance
        lr=5e-3,               # large step size to exaggerate changes
        steps=50,              # more iterations
        batch_size=16,          # small batch for quick feedback
        dp_clip=1000.0,
        dp_noise_multiplier=0.0 # turn off noise for testing
    )

    # Evaluate AFTER unlearning
    print(f"\n Client {cid} AFTER unlearning evaluations:")
    forget_loss_after, forget_ppl_after = c.evaluate_texts_return(forget_texts, label="Forget_AFTER")
    retain_loss_after, retain_ppl_after = c.evaluate_texts_return(retain_texts, label="Retain_AFTER")
    combined_loss_after, combined_ppl_after = c.evaluate_texts_return(forget_texts + retain_texts, label="Combined_AFTER")

    # Print a concise DataFrame table showing before/after metrics
    results_df = pd.DataFrame({
        "set": ["forget", "retain", "combined"],
        "loss_before": [forget_loss_before, retain_loss_before, combined_loss_before],
        "ppl_before": [forget_ppl_before, retain_ppl_before, combined_ppl_before],
        "loss_after": [forget_loss_after, retain_loss_after, combined_loss_after],
        "ppl_after": [forget_ppl_after, retain_ppl_after, combined_ppl_after],
    })

    print("\n Summary (before -> after):")
    with pd.option_context('display.float_format', '{:0.4f}'.format):
        print(results_df.to_string(index=False))

# -----------------------------
#  Save final global LoRA model (average client adapters)
# -----------------------------
ckpt_paths = [f"checkpoints/client{cid}_lora_last.pth" for cid in range(num_clients)]
existing_ckpts = [p for p in ckpt_paths if os.path.exists(p)]
if not existing_ckpts:
    print(" No client LoRA checkpoints found to average. Skipping final averaging.")
else:
    lora_ckpts = [torch.load(p, map_location="cpu") for p in existing_ckpts]
    all_keys = sorted(set().union(*[set(x.keys()) for x in lora_ckpts]))
    avg_state = {}
    for k in all_keys:
        stacked = torch.stack([ckpt[k].float() for ckpt in lora_ckpts], dim=0)
        avg_state[k] = stacked.mean(dim=0).to(device)

    final_global_model = create_lora_model()
    final_state = final_global_model.state_dict()
    for k, v in avg_state.items():
        if k in final_state:
            final_state[k] = v
    final_global_model.load_state_dict(final_state, strict=False)
    final_global_model.eval()
    print(" Final global LoRA model ready.")
    save_dir = "./final_global_model_2"
    final_global_model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"Final global LoRA model and tokenizer saved to {save_dir}")

In [ ]:
!pip install nbformat

In [ ]:

print("Manually saving a clean notebook...")

 
import IPython

# Get all code cells from the current runtime
code_cells = []
for cell in IPython.get_ipython().history_manager.input_hist_raw:
    if cell.strip():  # Skip empty cells
        code_cells.append(cell)

# Create a new notebook structure
notebook_content = {
    "cells": [],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python",
            "version": "3.8.0"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 0
}

# Add all code cells
for code in code_cells:
    notebook_content["cells"].append({
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [code]
    })

# Save the notebook
import json
with open('Federated_Unlearning_simple.ipynb', 'w') as f:
    json.dump(notebook_content, f)

# Download it
from google.colab import files
files.download('Federated_Unlearning_simple.ipynb')

print("Simple notebook downloaded. It contains all code cells without outputs.")